# Runner: Car Liability Model v1

**Purpose:** Orchestrate full pipeline execution using papermill

**Flow:**
1. Template 01: Data assembly (join master + aux, filter folds)
2. Template 02: Data conditioning (types, nulls, feature engineering)
3. Template 03: Verification (data quality checks)
4. Template 04: Model preparation (train/test split, DMatrix)
5. Template 05: Model training (XGBoost fit, predictions, metrics)

**Memory:** Each template runs in subprocess via papermill, cleans memory after checkpoint

In [1]:
import papermill as pm
import os
import sys
from datetime import datetime
import yaml
from pathlib import Path

# Detect project root and add lib to path
current_dir = Path.cwd()
if current_dir.name == 'runners':
    project_root = current_dir.parent
else:
    project_root = current_dir

# Add lib directory to Python path BEFORE importing
lib_path = str(project_root / 'lib')
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

# Now import
from utils import setup_notebook_environment

## Configuration

In [2]:
# Auto-detect project root (cross-machine compatible)
project_root = setup_notebook_environment()

print(f"Project root: {project_root}")
print(f"Current directory: {os.getcwd()}")

# Experiment configuration (relative to project root)
config_path = "config/car_liab/v1"
config_file = f"{config_path}/config.yaml"

# Load config to get output path
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

# Get current machine ID and output path
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'  # current.pc has '3', config key is 'PC3'
output_base = cfg['machines'][pc_id]['paths']['output_path']
experiment_name = cfg['experiment']['name']

print(f"\nMachine: {pc_id}")
print(f"Experiment: {experiment_name}")
print(f"Config: {config_path}")
print(f"Output: {output_base}")

Project root: /Users/Mach/dev/aps/code/26Dmodelv1
Current directory: /Users/Mach/dev/aps/code/26Dmodelv1

Machine: PC3
Experiment: car_liab_v1
Config: config/car_liab/v1
Output: output/car_liab/v1


In [3]:
# Create output directories
os.makedirs(f"{output_base}/notebooks", exist_ok=True)
os.makedirs(f"{output_base}/data", exist_ok=True)
os.makedirs(f"{output_base}/models", exist_ok=True)
os.makedirs(f"{output_base}/results", exist_ok=True)
os.makedirs(f"{output_base}/plots", exist_ok=True)
os.makedirs(f"{config_path}/config_generated", exist_ok=True)

print("Output directories created")

Output directories created


## Stage 00: EDA Setup (Manual)

In [4]:
# Copy EDA template (always update to latest version)
import shutil

eda_template = 'templates/00_eda_for_allfeatures.ipynb'
eda_output = f'{output_base}/notebooks/00_eda.ipynb'

# Always copy latest template
shutil.copy(eda_template, eda_output)
print(f"[x] Copied EDA template to: {eda_output}")

# Check if master encoding exists
master_encoding = f'{config_path}/config_generated/master_feature_encoding.csv'
if os.path.exists(master_encoding):
    print(f"[x] Master encoding exists: {master_encoding}")
else:
    print(f"[!] WARNING: Master encoding not found")
    print(f"    Run {eda_output} manually before pipeline")

[x] Copied EDA template to: output/car_liab/v1/notebooks/00_eda.ipynb
[x] Master encoding exists: config/car_liab/v1/config_generated/master_feature_encoding.csv


## Stage Configuration

In [5]:
# Define pipeline stages
stages = [
    {
        'name': 'stage_01',
        'title': 'Data Assembly',
        'template': 'templates/01_data_assembly.ipynb',
        'output': f'{output_base}/notebooks/01_data_assembly.ipynb',
        'enabled': cfg['execution']['stages']['stage_01']
    },
    {
        'name': 'stage_02',
        'title': 'Data Conditioning',
        'template': 'templates/02_data_conditioning.ipynb',
        'output': f'{output_base}/notebooks/02_data_conditioning.ipynb',
        'enabled': cfg['execution']['stages']['stage_02']
    },
    {
        'name': 'stage_03',
        'title': 'Verification',
        'template': 'templates/03_verification.ipynb',
        'output': f'{output_base}/notebooks/03_verification.ipynb',
        'enabled': cfg['execution']['stages']['stage_03']
    },
    {
        'name': 'stage_04a',
        'title': 'PCA Feature Engineering',
        'template': 'templates/04a_featureengineering.ipynb',
        'output': f'{output_base}/notebooks/04a_featureengineering.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_04a', True)
    },
    {
        'name': 'stage_04b',
        'title': 'Model Preparation',
        'template': 'templates/04b_train_test_split.ipynb',
        'output': f'{output_base}/notebooks/04b_train_test_split.ipynb',
        'enabled': cfg['execution']['stages']['stage_04b']
    },
    {
        'name': 'stage_04c',
        'title': 'Feature Encoding',
        'template': 'templates/04c_feature_encoding.ipynb',
        'output': f'{output_base}/notebooks/04c_feature_encoding.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_04c', True)
    },
    {
        'name': 'stage_05a',
        'title': 'Initial Model',
        'template': 'templates/05a_model_initial.ipynb',
        'output': f'{output_base}/notebooks/05a_model_initial.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_05a', False)
    },
    {
        'name': 'stage_05b',
        'title': 'HPO',
        'template': 'templates/05b_hpo.ipynb',
        'output': f'{output_base}/notebooks/05b_hpo.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_05b', False)
    },
    {
        'name': 'stage_05c',
        'title': 'Production Model',
        'template': 'templates/05c_production.ipynb',
        'output': f'{output_base}/notebooks/05c_production.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_05c', False)
    },
    {
        'name': 'stage_06',
        'title': 'SHAP Dataframe',
        'template': 'templates/06_shap_dataframe.ipynb',
        'output': f'{output_base}/notebooks/06_shap_dataframe.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_06', True)
    },
    {
        'name': 'stage_07',
        'title': 'SHAP Analysis',
        'template': 'templates/07_shap_analysis.ipynb',
        'output': f'{output_base}/notebooks/07_shap_analysis.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_07', True)
    },
    {
        'name': 'stage_08',
        'title': 'Scoring',
        'template': 'templates/08_scoring.ipynb',
        'output': f'{output_base}/notebooks/08_scoring.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_08', False)
    },
    {
        'name': 'stage_09',
        'title': 'State Analysis',
        'template': 'templates/09_state_analysis.ipynb',
        'output': f'{output_base}/notebooks/09_state_analysis.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_09', False)
    }
]

print("Pipeline stages:")
for stage in stages:
    status = "ENABLED" if stage['enabled'] else "DISABLED"
    print(f"  {stage['name']}: {stage['title']} - {status}")

Pipeline stages:
  stage_01: Data Assembly - ENABLED
  stage_02: Data Conditioning - ENABLED
  stage_03: Verification - ENABLED
  stage_04a: PCA Feature Engineering - ENABLED
  stage_04b: Model Preparation - ENABLED
  stage_04c: Feature Encoding - ENABLED
  stage_05a: Initial Model - ENABLED
  stage_05b: HPO - ENABLED
  stage_05c: Production Model - ENABLED
  stage_06: SHAP Dataframe - ENABLED
  stage_07: SHAP Analysis - ENABLED
  stage_08: Scoring - ENABLED
  stage_09: State Analysis - ENABLED


## Execute Pipeline

In [6]:
# Pipeline parameters (injected into each template)
parameters = {
    'config_path': config_path
}

print(f"\nParameters to inject:")
print(f"  config_path: {config_path}")


Parameters to inject:
  config_path: config/car_liab/v1


In [7]:
# Execute each stage
start_time = datetime.now()
print(f"\n{'='*60}")
print(f"PIPELINE EXECUTION START: {start_time}")
print(f"{'='*60}\n")

results = {}

for stage in stages:
    if not stage['enabled']:
        print(f"\n[SKIP] {stage['name']}: {stage['title']}")
        results[stage['name']] = 'skipped'
        continue
    
    print(f"\n{'#'*60}")
    print(f"# {stage['name'].upper()}: {stage['title'].upper()}")
    print(f"{'#'*60}")
    print(f"Template: {stage['template']}")
    print(f"Output:   {stage['output']}")
    print(f"\nExecuting via papermill...\n")
    
    try:
        stage_start = datetime.now()
        
        # Get kernel name from machine-specific conda environment
        kernel_name = cfg['machines'][pc_id].get('conda_env', 'python3')
        
        pm.execute_notebook(
            input_path=stage['template'],
            output_path=stage['output'],
            parameters=parameters,
            kernel_name=kernel_name
        )
        
        stage_end = datetime.now()
        duration = (stage_end - stage_start).total_seconds()
        
        print(f"\n[OK] {stage['name']} completed in {duration:.1f}s")
        results[stage['name']] = 'success'
        
    except Exception as e:
        stage_end = datetime.now()
        duration = (stage_end - stage_start).total_seconds()
        
        print(f"\n✗ {stage['name']} FAILED after {duration:.1f}s")
        print(f"Error: {str(e)}")
        results[stage['name']] = 'failed'
        
        if cfg['execution']['stop_on_error']:
            print(f"\nStopping pipeline (stop_on_error=true)")
            break

end_time = datetime.now()
total_duration = (end_time - start_time).total_seconds()

print(f"\n{'='*60}")
print(f"PIPELINE EXECUTION END: {end_time}")
print(f"Total duration: {total_duration:.1f}s ({total_duration/60:.1f}m)")
print(f"{'='*60}")


PIPELINE EXECUTION START: 2026-09-11 01:51:51.108723


############################################################
# STAGE_01: DATA ASSEMBLY
############################################################
Template: templates/01_data_assembly.ipynb
Output:   output/car_liab/v1/notebooks/01_data_assembly.ipynb

Executing via papermill...



Executing:   0%|          | 0/17 [00:00<?, ?cell/s]

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.



[OK] stage_01 completed in 39.5s

############################################################
# STAGE_02: DATA CONDITIONING
############################################################
Template: templates/02_data_conditioning.ipynb
Output:   output/car_liab/v1/notebooks/02_data_conditioning.ipynb

Executing via papermill...



Executing:   0%|          | 0/13 [00:00<?, ?cell/s]

...

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.



[OK] stage_07 completed in 47.0s

############################################################
# STAGE_08: SCORING
############################################################
Template: templates/08_scoring.ipynb
Output:   output/car_liab/v1/notebooks/08_scoring.ipynb

Executing via papermill...



Executing:   0%|          | 0/14 [00:00<?, ?cell/s]

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.



[OK] stage_08 completed in 373.6s

############################################################
# STAGE_09: STATE ANALYSIS
############################################################
Template: templates/09_state_analysis.ipynb
Output:   output/car_liab/v1/notebooks/09_state_analysis.ipynb

Executing via papermill...


✗ stage_09 FAILED after 0.0s
Error: No language found in notebook and no override provided.

Stopping pipeline (stop_on_error=true)

PIPELINE EXECUTION END: 2026-09-11 02:45:45.824514
Total duration: 3234.7s (53.9m)


## Summary

In [8]:
# Print summary
print("\n" + "="*60)
print("PIPELINE SUMMARY")
print("="*60)

success_count = sum(1 for v in results.values() if v == 'success')
failed_count = sum(1 for v in results.values() if v == 'failed')
skipped_count = sum(1 for v in results.values() if v == 'skipped')

for stage_name, status in results.items():
    emoji = '[OK]' if status == 'success' else '✗' if status == 'failed' else '-'
    print(f"{emoji} {stage_name}: {status}")

print(f"\nTotal: {len(results)} stages")
print(f"  Success: {success_count}")
print(f"  Failed:  {failed_count}")
print(f"  Skipped: {skipped_count}")
print("="*60)

if failed_count > 0:
    print("\n PIPELINE COMPLETED WITH ERRORS")
elif success_count > 0:
    print("\n PIPELINE COMPLETED SUCCESSFULLY")
else:
    print("\n NO STAGES EXECUTED")


PIPELINE SUMMARY
[OK] stage_01: success
[OK] stage_02: success
[OK] stage_03: success
[OK] stage_04a: success
[OK] stage_04b: success
[OK] stage_04c: success
[OK] stage_05a: success
[OK] stage_05b: success
[OK] stage_05c: success
[OK] stage_06: success
[OK] stage_07: success
[OK] stage_08: success
✗ stage_09: failed

Total: 13 stages
  Success: 12
  Failed:  1
  Skipped: 0

 PIPELINE COMPLETED WITH ERRORS


In [9]:
# Save execution log
log_data = {
    'experiment': experiment_name,
    'config_path': config_path,
    'start_time': start_time.isoformat(),
    'end_time': end_time.isoformat(),
    'duration_seconds': total_duration,
    'results': results
}

log_file = f"{output_base}/execution_log.yaml"
with open(log_file, 'w') as f:
    yaml.dump(log_data, f, default_flow_style=False)

print(f"\nExecution log saved: {log_file}")


Execution log saved: output/car_liab/v1/execution_log.yaml
